# TfL Delay Modeling: 60s vs 180s Targets

This notebook rebuilds the late-train classification workflow from the processed dataset and compares two target definitions:

- `late`: deviation from baseline greater than 60 seconds
- `late_3min`: deviation from baseline greater than 180 seconds

The goal is not to force the models into generic metric bands. Instead, we compare both targets using:

- class balance
- baseline comparisons
- threshold-free ranking metrics
- confusion-matrix outcomes at candidate thresholds
- alert burden metrics that better reflect product usability


In [1]:
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from xgboost import XGBClassifier

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    f1_score,
    log_loss,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

RANDOM_STATE = 42
USE_GPU_IF_AVAILABLE = True
THRESHOLD_GRID = np.round(np.linspace(0.10, 0.90, 17), 2)

PASSENGER_PRECISION_FLOOR = 0.60
OPS_PRECISION_FLOOR = 0.50

DATA_PATH = Path('../data/processed/dataset_192H_60.parquet')
if not DATA_PATH.exists():
    raise FileNotFoundError(DATA_PATH.resolve())

print('Setup complete.')


Setup complete.


In [2]:
df = pd.read_parquet(DATA_PATH)
df['observed_at'] = pd.to_datetime(df['observed_at'], utc=True, errors='coerce')
df = df.dropna(subset=['observed_at', 'deviation_from_baseline', 'late']).copy()

df['late'] = df['late'].astype(int)
df['late_3min'] = (df['deviation_from_baseline'] > 180).astype(int)

target_summary = pd.DataFrame([
    {
        'target': 'late',
        'positive_rate': float(df['late'].mean()),
        'positive_count': int(df['late'].sum()),
        'negative_count': int((1 - df['late']).sum()),
    },
    {
        'target': 'late_3min',
        'positive_rate': float(df['late_3min'].mean()),
        'positive_count': int(df['late_3min'].sum()),
        'negative_count': int((1 - df['late_3min']).sum()),
    },
])

print(f'Loaded {DATA_PATH} with shape {df.shape}')
display(target_summary.assign(positive_rate=lambda x: x['positive_rate'].round(4)))


Loaded ..\data\processed\dataset_192H_60.parquet with shape (24571322, 19)


,target,positive_rate,positive_count,negative_count
0,late,0.4447,10926397,13644925
1,late_3min,0.3566,8762850,15808472


In [3]:
# Keep the starting feature set aligned with models_kilian_v5_60_sec.ipynb for a fair comparison.
# We exclude `deviation_from_baseline` because it directly defines the targets.
# We also exclude `time_to_station` and `roll_std_tts_10m` in this first comparison notebook
# so we can isolate the target-definition question before changing the feature recipe.

categorical_features = [
    'stop_id', 'stop_name', 'line_id', 'direction', 'platform_name', 'destination_name'
]

numeric_features = [
    'hour', 'weekday', 'is_weekend',
    'roll_mean_tts_10m', 'roll_max_tts_10m', 'roll_count_10m', 'baseline_median_tts'
]

feature_cols = numeric_features + categorical_features
missing_cols = [c for c in feature_cols + ['observed_at', 'late', 'late_3min'] if c not in df.columns]
if missing_cols:
    raise ValueError(f'Missing required columns: {missing_cols}')

def make_preprocessor():
    return ColumnTransformer(
        transformers=[
            ('num', SimpleImputer(strategy='median'), numeric_features),
            ('cat', Pipeline(steps=[
                ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
                ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=True, dtype=np.float32, min_frequency=10)),
            ]), categorical_features),
        ],
        remainder='drop',
        sparse_threshold=1.0,
    )

def make_models(y_train):
    pos_count = int((y_train == 1).sum())
    neg_count = int((y_train == 0).sum())
    scale_pos_weight = max(1.0, neg_count / max(pos_count, 1))
    gpu_enabled = bool(USE_GPU_IF_AVAILABLE)

    return {
        'DummyPrior': DummyClassifier(strategy='prior'),
        'AlwaysLate': DummyClassifier(strategy='constant', constant=1),
        'AlwaysNotLate': DummyClassifier(strategy='constant', constant=0),
        'LogisticRegression': LogisticRegression(max_iter=2000, class_weight='balanced', n_jobs=None),
        'XGBoost': XGBClassifier(
            n_estimators=260,
            max_depth=6,
            learning_rate=0.05,
            subsample=0.85,
            colsample_bytree=0.80,
            min_child_weight=3,
            reg_lambda=2.0,
            gamma=0.1,
            scale_pos_weight=scale_pos_weight,
            eval_metric='logloss',
            tree_method='hist',
            device='cuda' if gpu_enabled else 'cpu',
            n_jobs=-1,
            random_state=RANDOM_STATE,
            verbosity=0,
        ),
        'LightGBM': lgb.LGBMClassifier(
            n_estimators=300,
            max_depth=7,
            learning_rate=0.05,
            num_leaves=63,
            subsample=0.85,
            colsample_bytree=0.85,
            min_child_samples=30,
            reg_lambda=2.0,
            class_weight='balanced',
            n_jobs=-1,
            random_state=RANDOM_STATE,
            verbosity=-1,
            device_type='gpu' if gpu_enabled else 'cpu',
        ),
        'CatBoost': CatBoostClassifier(
            iterations=400,
            depth=8,
            learning_rate=0.05,
            l2_leaf_reg=5.0,
            eval_metric='Logloss',
            auto_class_weights='Balanced',
            random_seed=RANDOM_STATE,
            verbose=False,
            task_type='GPU' if gpu_enabled else 'CPU',
        ),
    }

def fit_with_fallback(model_name, model, X_train_t, y_train):
    try:
        model.fit(X_train_t, y_train)
        return model, None
    except Exception as exc:
        if model_name == 'XGBoost':
            model.set_params(device='cpu')
            model.fit(X_train_t, y_train)
            return model, f'GPU failed, used CPU: {exc}'
        if model_name == 'LightGBM':
            model.set_params(device_type='cpu')
            model.fit(X_train_t, y_train)
            return model, f'GPU failed, used CPU: {exc}'
        if model_name == 'CatBoost':
            model.set_params(task_type='CPU')
            model.fit(X_train_t, y_train)
            return model, f'GPU failed, used CPU: {exc}'
        raise

def collect_probability_metrics(y_true, y_prob):
    return {
        'logloss': float(log_loss(y_true, y_prob, labels=[0, 1])),
        'avg_precision': float(average_precision_score(y_true, y_prob)),
        'roc_auc': float(roc_auc_score(y_true, y_prob)),
    }

def threshold_rows(y_true, y_prob, observed_at, stop_id, grid):
    rows = []
    hours = max((observed_at.max() - observed_at.min()).total_seconds() / 3600.0, 1e-9)
    station_hours = max(int(pd.DataFrame({
        'stop_id': stop_id.astype(str).to_numpy(),
        'hour_bucket': observed_at.dt.floor('h').astype(str).to_numpy(),
    }).drop_duplicates().shape[0]), 1)

    for threshold in grid:
        y_pred = (y_prob >= threshold).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
        precision = precision_score(y_true, y_pred, zero_division=0)
        recall = recall_score(y_true, y_pred, zero_division=0)
        rows.append({
            'threshold': float(threshold),
            'precision': float(precision),
            'recall': float(recall),
            'f1': float(f1_score(y_true, y_pred, zero_division=0)),
            'tp': int(tp),
            'fp': int(fp),
            'tn': int(tn),
            'fn': int(fn),
            'specificity': float(tn / max(tn + fp, 1)),
            'alert_rate': float(y_pred.mean()),
            'alerts_per_hour': float(y_pred.sum() / hours),
            'false_alerts_per_hour': float(fp / hours),
            'alerts_per_station_hour': float(y_pred.sum() / station_hours),
            'false_alerts_per_station_hour': float(fp / station_hours),
        })

    return pd.DataFrame(rows)

def pick_operating_point(threshold_df, precision_floor):
    feasible = threshold_df[threshold_df['precision'] >= precision_floor].copy()
    if len(feasible) > 0:
        pick = feasible.sort_values(['recall', 'f1', 'precision'], ascending=False).iloc[0].copy()
        pick['meets_precision_floor'] = True
    else:
        pick = threshold_df.sort_values(['precision', 'f1', 'recall'], ascending=False).iloc[0].copy()
        pick['meets_precision_floor'] = False
    return pick

def evaluate_target(df_all, target_col, run_backtest=False):
    frame = df_all[feature_cols + ['observed_at', target_col]].copy()
    frame = frame.sort_values('observed_at').reset_index(drop=True)
    X = frame[feature_cols].copy()
    y = frame[target_col].astype(int).copy()
    observed_at = frame['observed_at'].copy()
    stop_ids = X['stop_id'].copy()

    tscv = TimeSeriesSplit(n_splits=5)
    splits = list(tscv.split(X))
    train_idx, test_idx = splits[-1]

    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    observed_test = observed_at.iloc[test_idx]
    stop_test = stop_ids.iloc[test_idx]

    preprocessor = make_preprocessor()
    X_train_t = preprocessor.fit_transform(X_train).astype(np.float32)
    X_test_t = preprocessor.transform(X_test).astype(np.float32)

    models = make_models(y_train)
    metric_rows = []
    threshold_tables = {}
    passenger_rows = []
    ops_rows = []
    fitted_models = {}

    for model_name, model in models.items():
        fitted, note = fit_with_fallback(model_name, model, X_train_t, y_train)
        fitted_models[model_name] = fitted
        y_prob = fitted.predict_proba(X_test_t)[:, 1]
        metric_row = {
            'target': target_col,
            'model': model_name,
            'train_positive_rate': float(y_train.mean()),
            'test_positive_rate': float(y_test.mean()),
            'fit_note': note,
            **collect_probability_metrics(y_test, y_prob),
        }
        metric_rows.append(metric_row)

        table = threshold_rows(y_test, y_prob, observed_test, stop_test, THRESHOLD_GRID)
        threshold_tables[model_name] = table

        passenger_pick = pick_operating_point(table, PASSENGER_PRECISION_FLOOR)
        passenger_pick['target'] = target_col
        passenger_pick['model'] = model_name
        passenger_pick['mode'] = 'passenger'
        passenger_pick['precision_floor'] = PASSENGER_PRECISION_FLOOR
        passenger_rows.append(passenger_pick)

        ops_pick = pick_operating_point(table, OPS_PRECISION_FLOOR)
        ops_pick['target'] = target_col
        ops_pick['model'] = model_name
        ops_pick['mode'] = 'ops'
        ops_pick['precision_floor'] = OPS_PRECISION_FLOOR
        ops_rows.append(ops_pick)

    metrics = pd.DataFrame(metric_rows)
    dummy_row = metrics.loc[metrics['model'] == 'DummyPrior'].iloc[0]
    metrics['lift_vs_dummy_avg_precision_abs'] = metrics['avg_precision'] - float(dummy_row['avg_precision'])
    metrics['lift_vs_dummy_roc_auc_abs'] = metrics['roc_auc'] - float(dummy_row['roc_auc'])
    metrics = metrics.sort_values(['avg_precision', 'roc_auc'], ascending=False).reset_index(drop=True)

    backtest = None
    if run_backtest:
        bt_rows = []
        fold_splitter = TimeSeriesSplit(n_splits=4)
        tree_models = ['XGBoost', 'LightGBM', 'CatBoost']
        for fold_num, (tr_idx, te_idx) in enumerate(fold_splitter.split(X), start=1):
            X_tr, X_te = X.iloc[tr_idx], X.iloc[te_idx]
            y_tr, y_te = y.iloc[tr_idx], y.iloc[te_idx]
            obs_te = observed_at.iloc[te_idx]
            stop_te = stop_ids.iloc[te_idx]
            fold_pre = make_preprocessor()
            X_tr_t = fold_pre.fit_transform(X_tr).astype(np.float32)
            X_te_t = fold_pre.transform(X_te).astype(np.float32)
            fold_models = make_models(y_tr)
            for model_name in tree_models:
                try:
                    fitted, note = fit_with_fallback(model_name, clone(fold_models[model_name]), X_tr_t, y_tr)
                    prob = fitted.predict_proba(X_te_t)[:, 1]
                    table = threshold_rows(y_te, prob, obs_te, stop_te, THRESHOLD_GRID)
                    pick = pick_operating_point(table, OPS_PRECISION_FLOOR)
                    bt_rows.append({
                        'target': target_col,
                        'fold': fold_num,
                        'model': model_name,
                        'threshold': float(pick['threshold']),
                        'precision': float(pick['precision']),
                        'recall': float(pick['recall']),
                        'f1': float(pick['f1']),
                        'avg_precision': float(average_precision_score(y_te, prob)),
                        'roc_auc': float(roc_auc_score(y_te, prob)),
                        'fit_note': note,
                    })
                except Exception as exc:
                    bt_rows.append({
                        'target': target_col,
                        'fold': fold_num,
                        'model': model_name,
                        'error': str(exc),
                    })
        backtest = pd.DataFrame(bt_rows)

    return {
        'target': target_col,
        'metrics': metrics,
        'threshold_tables': threshold_tables,
        'passenger_recommendations': pd.DataFrame(passenger_rows).sort_values(['precision', 'recall'], ascending=False).reset_index(drop=True),
        'ops_recommendations': pd.DataFrame(ops_rows).sort_values(['precision', 'recall'], ascending=False).reset_index(drop=True),
        'backtest': backtest,
    }

print('Helper functions ready.')


Helper functions ready.


In [4]:
results_60 = evaluate_target(df, 'late', run_backtest=False)
results_180 = evaluate_target(df, 'late_3min', run_backtest=False)

print('Probability metrics for 60-second target')
display(results_60['metrics'][[
    'target', 'model', 'train_positive_rate', 'test_positive_rate',
    'avg_precision', 'roc_auc', 'logloss',
    'lift_vs_dummy_avg_precision_abs', 'lift_vs_dummy_roc_auc_abs'
]].round(4))

print('Passenger-oriented operating points for 60-second target')
display(results_60['passenger_recommendations'][[
    'target', 'model', 'threshold', 'precision', 'recall', 'f1',
    'alert_rate', 'alerts_per_hour', 'false_alerts_per_hour',
    'alerts_per_station_hour', 'false_alerts_per_station_hour',
    'meets_precision_floor'
]].round(4))

print('Probability metrics for 180-second target')
display(results_180['metrics'][[
    'target', 'model', 'train_positive_rate', 'test_positive_rate',
    'avg_precision', 'roc_auc', 'logloss',
    'lift_vs_dummy_avg_precision_abs', 'lift_vs_dummy_roc_auc_abs'
]].round(4))

print('Passenger-oriented operating points for 180-second target')
display(results_180['passenger_recommendations'][[
    'target', 'model', 'threshold', 'precision', 'recall', 'f1',
    'alert_rate', 'alerts_per_hour', 'false_alerts_per_hour',
    'alerts_per_station_hour', 'false_alerts_per_station_hour',
    'meets_precision_floor'
]].round(4))


e:\dsbootcamp_neuefische\capstone_project\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 2000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
e:\dsbootcamp_neuefische\capstone_project\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
e:\dsbootcamp_neuefische\capstone_project\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 2000 iteration(

Probability metrics for 60-second target


,target,model,train_positive_rate,test_positive_rate,avg_precision,roc_auc,logloss,lift_vs_dummy_avg_precision_abs,lift_vs_dummy_roc_auc_abs
0,late,LightGBM,0.4443,0.4465,0.4854,0.5477,0.6849,0.0389,0.0477
1,late,CatBoost,0.4443,0.4465,0.4834,0.5438,0.6863,0.0368,0.0438
2,late,XGBoost,0.4443,0.4465,0.4820,0.5446,0.6855,0.0355,0.0446
3,late,LogisticRegression,0.4443,0.4465,0.4717,0.5370,0.6900,0.0252,0.0370
4,late,DummyPrior,0.4443,0.4465,0.4465,0.5000,0.6874,0.0000,0.0000
5,late,AlwaysLate,0.4443,0.4465,0.4465,0.5000,19.9492,0.0000,0.0000
6,late,AlwaysNotLate,0.4443,0.4465,0.4465,0.5000,16.0945,0.0000,0.0000


Passenger-oriented operating points for 60-second target


,target,model,threshold,precision,recall,f1,alert_rate,alerts_per_hour,false_alerts_per_hour,alerts_per_station_hour,false_alerts_per_station_hour,meets_precision_floor
0,late,LightGBM,0.65,0.6846,0.0027,0.0054,0.0018,277.8429,87.6360,6.6676,2.1031,True
1,late,XGBoost,0.65,0.6629,0.0024,0.0048,0.0016,259.6192,87.5199,6.2303,2.1003,True
2,late,CatBoost,0.65,0.6604,0.0028,0.0056,0.0019,300.1678,101.9518,7.2033,2.4466,True
3,late,LogisticRegression,0.70,0.6034,0.0006,0.0011,0.0004,66.6266,26.4262,1.5989,0.6342,True
4,late,DummyPrior,0.10,0.4465,1.0000,0.6174,1.0000,158449.7437,87697.5118,3802.4327,2104.5404,False
5,late,AlwaysLate,0.10,0.4465,1.0000,0.6174,1.0000,158449.7437,87697.5118,3802.4327,2104.5404,False
6,late,AlwaysNotLate,0.10,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,False


Probability metrics for 180-second target


,target,model,train_positive_rate,test_positive_rate,avg_precision,roc_auc,logloss,lift_vs_dummy_avg_precision_abs,lift_vs_dummy_roc_auc_abs
0,late_3min,LightGBM,0.3559,0.3602,0.4186,0.5826,0.6663,0.0584,0.0826
1,late_3min,CatBoost,0.3559,0.3602,0.4174,0.5792,0.6649,0.0572,0.0792
2,late_3min,XGBoost,0.3559,0.3602,0.4158,0.5835,0.6683,0.0556,0.0835
3,late_3min,LogisticRegression,0.3559,0.3602,0.3991,0.5669,0.6823,0.0389,0.0669
4,late_3min,DummyPrior,0.3559,0.3602,0.3602,0.5000,0.6536,0.0000,0.0000
5,late_3min,AlwaysLate,0.3559,0.3602,0.3602,0.5000,23.0621,0.0000,0.0000
6,late_3min,AlwaysNotLate,0.3559,0.3602,0.3602,0.5000,12.9816,0.0000,0.0000


Passenger-oriented operating points for 180-second target


,target,model,threshold,precision,recall,f1,alert_rate,alerts_per_hour,false_alerts_per_hour,alerts_per_station_hour,false_alerts_per_station_hour,meets_precision_floor
0,late_3min,LightGBM,0.7,0.7258,0.0010,0.0021,0.0005,80.9811,22.2089,1.9434,0.5330,True
1,late_3min,CatBoost,0.7,0.6788,0.0010,0.0021,0.0006,87.3265,28.0513,2.0956,0.6732,True
2,late_3min,LogisticRegression,0.8,0.6512,0.0000,0.0000,0.0000,1.6637,0.5804,0.0399,0.0139,True
3,late_3min,XGBoost,0.7,0.6051,0.0012,0.0023,0.0007,110.0383,43.4504,2.6407,1.0427,True
4,late_3min,DummyPrior,0.1,0.3602,1.0000,0.5296,1.0000,158449.7437,101381.9591,3802.4327,2432.9359,False
5,late_3min,AlwaysLate,0.1,0.3602,1.0000,0.5296,1.0000,158449.7437,101381.9591,3802.4327,2432.9359,False
6,late_3min,AlwaysNotLate,0.1,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,False


In [5]:
compare_targets = pd.concat([
    results_60['passenger_recommendations'],
    results_180['passenger_recommendations'],
], ignore_index=True)

compare_targets = compare_targets[[
    'target', 'model', 'threshold', 'precision', 'recall', 'f1',
    'alert_rate', 'false_alerts_per_hour', 'false_alerts_per_station_hour',
    'meets_precision_floor'
]].sort_values(['target', 'precision', 'recall'], ascending=[True, False, False]).reset_index(drop=True)

print('Passenger-facing comparison across both targets')
display(compare_targets.round(4))

best_by_target = compare_targets.groupby('target', as_index=False).first()
print('Top passenger-facing candidate per target')
display(best_by_target.round(4))


Passenger-facing comparison across both targets


,target,model,threshold,precision,recall,f1,alert_rate,false_alerts_per_hour,false_alerts_per_station_hour,meets_precision_floor
0,late,LightGBM,0.65,0.6846,0.0027,0.0054,0.0018,87.6360,2.1031,True
1,late,XGBoost,0.65,0.6629,0.0024,0.0048,0.0016,87.5199,2.1003,True
2,late,CatBoost,0.65,0.6604,0.0028,0.0056,0.0019,101.9518,2.4466,True
3,late,LogisticRegression,0.70,0.6034,0.0006,0.0011,0.0004,26.4262,0.6342,True
4,late,DummyPrior,0.10,0.4465,1.0000,0.6174,1.0000,87697.5118,2104.5404,False
5,late,AlwaysLate,0.10,0.4465,1.0000,0.6174,1.0000,87697.5118,2104.5404,False
6,late,AlwaysNotLate,0.10,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,False
7,late_3min,LightGBM,0.70,0.7258,0.0010,0.0021,0.0005,22.2089,0.5330,True
8,late_3min,CatBoost,0.70,0.6788,0.0010,0.0021,0.0006,28.0513,0.6732,True
9,late_3min,LogisticRegression,0.80,0.6512,0.0000,0.0000,0.0000,0.5804,0.0139,True


Top passenger-facing candidate per target


,target,model,threshold,precision,recall,f1,alert_rate,false_alerts_per_hour,false_alerts_per_station_hour,meets_precision_floor
0,late,LightGBM,0.65,0.6846,0.0027,0.0054,0.0018,87.6360,2.1031,True
1,late_3min,LightGBM,0.70,0.7258,0.0010,0.0021,0.0005,22.2089,0.5330,True


## How to interpret the outputs

Use the notebook in this order:

1. Check class balance for `late` and `late_3min`.
2. Compare `avg_precision` and `roc_auc` against the dummy baseline for each target.
3. Look at the passenger-oriented operating points first.
4. If none of the models reach the passenger precision floor, the target is likely too noisy or the features are not strong enough yet.
5. Compare alert burden. Even a model with acceptable precision may still generate too many false alerts per hour.

If the 180-second target gives materially better precision and cleaner alert burden than the 60-second target, that is evidence that the 180-second label may be the better product target.


## Next Iteration: Focus On `late_3min`

This appended section keeps the results above intact and moves to the next experiment direction:

- keep `late_3min` as the main target
- drop `AlwaysLate` and `AlwaysNotLate` from routine runs
- keep routine comparisons focused on `DummyPrior`, `LogisticRegression`, `XGBoost`, and `LightGBM`
- run one model per cell for easier runtime tracking
- test whether feature improvements can recover useful recall without losing too much precision

Important note on `time_to_station`:

- `deviation_from_baseline` is direct leakage and must stay excluded
- `time_to_station` is not direct leakage by itself, because it is available at inference time and is not identical to the target
- however, it is a strong proxy and should be treated as a deliberate feature experiment, not silently added without review


In [6]:
MAIN_TARGET = 'late_3min'

enhanced_numeric_features = [
    'hour', 'weekday', 'is_weekend',
    'time_to_station',
    'roll_mean_tts_10m', 'roll_max_tts_10m', 'roll_count_10m', 'roll_std_tts_10m',
    'baseline_median_tts',
]
enhanced_numeric_features = [c for c in enhanced_numeric_features if c in df.columns]

feature_sets = {
    'baseline_v5': {
        'numeric': numeric_features,
        'categorical': categorical_features,
    },
    'enhanced_v1': {
        'numeric': enhanced_numeric_features,
        'categorical': categorical_features,
    },
}

routine_models = ['DummyPrior', 'LogisticRegression', 'XGBoost', 'LightGBM']

print('Next-iteration target:', MAIN_TARGET)
print('Feature sets:', list(feature_sets.keys()))
display(pd.DataFrame({
    'feature_set': list(feature_sets.keys()),
    'numeric_features': [', '.join(v['numeric']) for v in feature_sets.values()],
    'categorical_features': [', '.join(v['categorical']) for v in feature_sets.values()],
}))


Next-iteration target: late_3min
Feature sets: ['baseline_v5', 'enhanced_v1']


,feature_set,numeric_features,categorical_features
0,baseline_v5,"hour, weekday, is_weekend, roll_mean_tts_10m, ...","stop_id, stop_name, line_id, direction, platfo..."
1,enhanced_v1,"hour, weekday, is_weekend, time_to_station, ro...","stop_id, stop_name, line_id, direction, platfo..."


In [7]:
def make_preprocessor_for(numeric_cols, categorical_cols):
    return ColumnTransformer(
        transformers=[
            ('num', SimpleImputer(strategy='median'), numeric_cols),
            ('cat', Pipeline(steps=[
                ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
                ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=True, dtype=np.float32, min_frequency=10)),
            ]), categorical_cols),
        ],
        remainder='drop',
        sparse_threshold=1.0,
    )

def evaluate_one_model(df_all, target_col, model_name, numeric_cols, categorical_cols, precision_floor=PASSENGER_PRECISION_FLOOR):
    cols = numeric_cols + categorical_cols
    frame = df_all[cols + ['observed_at', target_col]].copy().sort_values('observed_at').reset_index(drop=True)
    X = frame[cols].copy()
    y = frame[target_col].astype(int).copy()
    observed_at = frame['observed_at'].copy()
    stop_ids = X['stop_id'].copy()

    tscv = TimeSeriesSplit(n_splits=5)
    train_idx, test_idx = list(tscv.split(X))[-1]
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    observed_test = observed_at.iloc[test_idx]
    stop_test = stop_ids.iloc[test_idx]

    preprocessor = make_preprocessor_for(numeric_cols, categorical_cols)
    X_train_t = preprocessor.fit_transform(X_train).astype(np.float32)
    X_test_t = preprocessor.transform(X_test).astype(np.float32)

    model_pool = make_models(y_train)
    model = clone(model_pool[model_name]) if model_name in ['XGBoost', 'LightGBM', 'CatBoost'] else model_pool[model_name]
    fitted, note = fit_with_fallback(model_name, model, X_train_t, y_train)
    y_prob = fitted.predict_proba(X_test_t)[:, 1]

    metrics_row = {
        'target': target_col,
        'model': model_name,
        'train_positive_rate': float(y_train.mean()),
        'test_positive_rate': float(y_test.mean()),
        'fit_note': note,
        **collect_probability_metrics(y_test, y_prob),
    }

    threshold_table = threshold_rows(y_test, y_prob, observed_test, stop_test, THRESHOLD_GRID)
    passenger_pick = pick_operating_point(threshold_table, precision_floor)
    passenger_pick['target'] = target_col
    passenger_pick['model'] = model_name
    passenger_pick['precision_floor'] = precision_floor

    return metrics_row, passenger_pick, threshold_table

def run_model_across_feature_sets(df_all, target_col, model_name, feature_set_map):
    metric_rows = []
    pick_rows = []
    threshold_tables = {}
    for feature_set_name, spec in feature_set_map.items():
        metrics_row, passenger_pick, threshold_table = evaluate_one_model(
            df_all=df_all,
            target_col=target_col,
            model_name=model_name,
            numeric_cols=spec['numeric'],
            categorical_cols=spec['categorical'],
        )
        metrics_row['feature_set'] = feature_set_name
        passenger_pick['feature_set'] = feature_set_name
        metric_rows.append(metrics_row)
        pick_rows.append(passenger_pick)
        threshold_tables[feature_set_name] = threshold_table

    metrics_df = pd.DataFrame(metric_rows).sort_values(['avg_precision', 'roc_auc'], ascending=False).reset_index(drop=True)
    picks_df = pd.DataFrame(pick_rows).sort_values(['precision', 'recall'], ascending=False).reset_index(drop=True)
    return {
        'metrics': metrics_df,
        'picks': picks_df,
        'threshold_tables': threshold_tables,
    }

print('Single-model experiment helpers ready.')


Single-model experiment helpers ready.


In [8]:
dummy_experiment = run_model_across_feature_sets(df, MAIN_TARGET, 'DummyPrior', feature_sets)

print('DummyPrior metrics across feature sets')
display(dummy_experiment['metrics'][[
    'feature_set', 'target', 'model', 'avg_precision', 'roc_auc', 'logloss'
]].round(4))

print('DummyPrior passenger-oriented operating points')
display(dummy_experiment['picks'][[
    'feature_set', 'threshold', 'precision', 'recall', 'f1',
    'false_alerts_per_hour', 'false_alerts_per_station_hour', 'meets_precision_floor'
]].round(4))


e:\dsbootcamp_neuefische\capstone_project\.venv\Lib\site-packages\sklearn\impute\_base.py:641: UserWarning: Skipping features without any observed values: ['roll_std_tts_10m']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
e:\dsbootcamp_neuefische\capstone_project\.venv\Lib\site-packages\sklearn\impute\_base.py:641: UserWarning: Skipping features without any observed values: ['roll_std_tts_10m']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(


DummyPrior metrics across feature sets


,feature_set,target,model,avg_precision,roc_auc,logloss
0,baseline_v5,late_3min,DummyPrior,0.3602,0.5,0.6536
1,enhanced_v1,late_3min,DummyPrior,0.3602,0.5,0.6536


DummyPrior passenger-oriented operating points


,feature_set,threshold,precision,recall,f1,false_alerts_per_hour,false_alerts_per_station_hour,meets_precision_floor
0,baseline_v5,0.1,0.3602,1.0,0.5296,101381.9591,2432.9359,False
1,enhanced_v1,0.1,0.3602,1.0,0.5296,101381.9591,2432.9359,False


In [9]:
logreg_experiment = run_model_across_feature_sets(df, MAIN_TARGET, 'LogisticRegression', feature_sets)

print('LogisticRegression metrics across feature sets')
display(logreg_experiment['metrics'][[
    'feature_set', 'target', 'model', 'avg_precision', 'roc_auc', 'logloss', 'fit_note'
]].round(4))

print('LogisticRegression passenger-oriented operating points')
display(logreg_experiment['picks'][[
    'feature_set', 'threshold', 'precision', 'recall', 'f1',
    'false_alerts_per_hour', 'false_alerts_per_station_hour', 'meets_precision_floor'
]].round(4))


e:\dsbootcamp_neuefische\capstone_project\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 2000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
e:\dsbootcamp_neuefische\capstone_project\.venv\Lib\site-packages\sklearn\impute\_base.py:641: UserWarning: Skipping features without any observed values: ['roll_std_tts_10m']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
e:\dsbootcamp_neuefische\capstone_project\.venv\Lib\site-packages\sklearn\impute\_base.py:641: UserWarning: Skipp

LogisticRegression metrics across feature sets


,feature_set,target,model,avg_precision,roc_auc,logloss,fit_note
0,enhanced_v1,late_3min,LogisticRegression,1.0000,1.0000,0.0045,None
1,baseline_v5,late_3min,LogisticRegression,0.3991,0.5669,0.6823,None


LogisticRegression passenger-oriented operating points


,feature_set,threshold,precision,recall,f1,false_alerts_per_hour,false_alerts_per_station_hour,meets_precision_floor
0,enhanced_v1,0.1,0.9864,1.0,0.9931,789.3817,18.9434,True
1,baseline_v5,0.8,0.6512,0.0,0.0000,0.5804,0.0139,True


In [10]:
xgb_experiment = run_model_across_feature_sets(df, MAIN_TARGET, 'XGBoost', feature_sets)

print('XGBoost metrics across feature sets')
display(xgb_experiment['metrics'][[
    'feature_set', 'target', 'model', 'avg_precision', 'roc_auc', 'logloss', 'fit_note'
]].round(4))

print('XGBoost passenger-oriented operating points')
display(xgb_experiment['picks'][[
    'feature_set', 'threshold', 'precision', 'recall', 'f1',
    'false_alerts_per_hour', 'false_alerts_per_station_hour', 'meets_precision_floor'
]].round(4))


e:\dsbootcamp_neuefische\capstone_project\.venv\Lib\site-packages\sklearn\impute\_base.py:641: UserWarning: Skipping features without any observed values: ['roll_std_tts_10m']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
e:\dsbootcamp_neuefische\capstone_project\.venv\Lib\site-packages\sklearn\impute\_base.py:641: UserWarning: Skipping features without any observed values: ['roll_std_tts_10m']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(


XGBoost metrics across feature sets


,feature_set,target,model,avg_precision,roc_auc,logloss,fit_note
0,enhanced_v1,late_3min,XGBoost,1.0000,1.0000,0.0087,None
1,baseline_v5,late_3min,XGBoost,0.4158,0.5835,0.6683,None


XGBoost passenger-oriented operating points


,feature_set,threshold,precision,recall,f1,false_alerts_per_hour,false_alerts_per_station_hour,meets_precision_floor
0,enhanced_v1,0.15,0.9779,1.0000,0.9888,1287.6881,30.9016,True
1,baseline_v5,0.70,0.6051,0.0012,0.0023,43.4504,1.0427,True


In [11]:
lgbm_experiment = run_model_across_feature_sets(df, MAIN_TARGET, 'LightGBM', feature_sets)

print('LightGBM metrics across feature sets')
display(lgbm_experiment['metrics'][[
    'feature_set', 'target', 'model', 'avg_precision', 'roc_auc', 'logloss', 'fit_note'
]].round(4))

print('LightGBM passenger-oriented operating points')
display(lgbm_experiment['picks'][[
    'feature_set', 'threshold', 'precision', 'recall', 'f1',
    'false_alerts_per_hour', 'false_alerts_per_station_hour', 'meets_precision_floor'
]].round(4))


e:\dsbootcamp_neuefische\capstone_project\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
e:\dsbootcamp_neuefische\capstone_project\.venv\Lib\site-packages\sklearn\impute\_base.py:641: UserWarning: Skipping features without any observed values: ['roll_std_tts_10m']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
e:\dsbootcamp_neuefische\capstone_project\.venv\Lib\site-packages\sklearn\impute\_base.py:641: UserWarning: Skipping features without any observed values: ['roll_std_tts_10m']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
e:\dsbootcamp_neuefische\capstone_project\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


LightGBM metrics across feature sets


,feature_set,target,model,avg_precision,roc_auc,logloss,fit_note
0,enhanced_v1,late_3min,LightGBM,1.0000,1.0000,0.0059,"GPU failed, used CPU: Check failed: (best_spli..."
1,baseline_v5,late_3min,LightGBM,0.4186,0.5826,0.6663,"GPU failed, used CPU: Check failed: (best_spli..."


LightGBM passenger-oriented operating points


,feature_set,threshold,precision,recall,f1,false_alerts_per_hour,false_alerts_per_station_hour,meets_precision_floor
0,enhanced_v1,0.1,0.9815,1.000,0.9906,1077.5551,25.8589,True
1,baseline_v5,0.7,0.7258,0.001,0.0021,22.2089,0.5330,True


In [12]:
next_iteration_summary = pd.concat([
    dummy_experiment['metrics'],
    logreg_experiment['metrics'],
    xgb_experiment['metrics'],
    lgbm_experiment['metrics'],
], ignore_index=True).sort_values(['feature_set', 'avg_precision', 'roc_auc'], ascending=[True, False, False]).reset_index(drop=True)

next_iteration_picks = pd.concat([
    dummy_experiment['picks'],
    logreg_experiment['picks'],
    xgb_experiment['picks'],
    lgbm_experiment['picks'],
], ignore_index=True).sort_values(['feature_set', 'precision', 'recall'], ascending=[True, False, False]).reset_index(drop=True)

print('Next-iteration summary metrics')
display(next_iteration_summary[[
    'feature_set', 'target', 'model', 'avg_precision', 'roc_auc', 'logloss'
]].round(4))

print('Next-iteration passenger-oriented picks')
display(next_iteration_picks[[
    'feature_set', 'model', 'threshold', 'precision', 'recall', 'f1',
    'false_alerts_per_hour', 'false_alerts_per_station_hour', 'meets_precision_floor'
]].round(4))


Next-iteration summary metrics


,feature_set,target,model,avg_precision,roc_auc,logloss
0,baseline_v5,late_3min,LightGBM,0.4186,0.5826,0.6663
1,baseline_v5,late_3min,XGBoost,0.4158,0.5835,0.6683
2,baseline_v5,late_3min,LogisticRegression,0.3991,0.5669,0.6823
3,baseline_v5,late_3min,DummyPrior,0.3602,0.5000,0.6536
4,enhanced_v1,late_3min,LightGBM,1.0000,1.0000,0.0059
5,enhanced_v1,late_3min,LogisticRegression,1.0000,1.0000,0.0045
6,enhanced_v1,late_3min,XGBoost,1.0000,1.0000,0.0087
7,enhanced_v1,late_3min,DummyPrior,0.3602,0.5000,0.6536


Next-iteration passenger-oriented picks


,feature_set,model,threshold,precision,recall,f1,false_alerts_per_hour,false_alerts_per_station_hour,meets_precision_floor
0,baseline_v5,LightGBM,0.70,0.7258,0.0010,0.0021,22.2089,0.5330,True
1,baseline_v5,LogisticRegression,0.80,0.6512,0.0000,0.0000,0.5804,0.0139,True
2,baseline_v5,XGBoost,0.70,0.6051,0.0012,0.0023,43.4504,1.0427,True
3,baseline_v5,DummyPrior,0.10,0.3602,1.0000,0.5296,101381.9591,2432.9359,False
4,enhanced_v1,LogisticRegression,0.10,0.9864,1.0000,0.9931,789.3817,18.9434,True
5,enhanced_v1,LightGBM,0.10,0.9815,1.0000,0.9906,1077.5551,25.8589,True
6,enhanced_v1,XGBoost,0.15,0.9779,1.0000,0.9888,1287.6881,30.9016,True
7,enhanced_v1,DummyPrior,0.10,0.3602,1.0000,0.5296,101381.9591,2432.9359,False
